# TP1 — Manipulation de base d'images numériques

**Vision par Ordinateur**

Ce notebook répond aux exercices 1 à 8 du TP :
1. Chargement et affichage d'images
2. Quantification et binarisation
3. Histogramme et contraste
4. Égalisation d'histogramme
5. Espaces colorimétriques
6. Transformations géométriques
7. Filtrage linéaire
8. Filtrage non linéaire

> Placez les images du TP dans le même dossier que ce notebook. Pour l'image en niveaux de gris, le notebook cherche automatiquement `mountain_gray.jpg`, `mountain_gray.jpeg`, `mountain_gray.png` ou `mountain_gray.avif`. Pour l'image couleur, il cherche `flowers.jpg`, `flowers.png`, `peppers.png` ou `peppers.jpg`.

## 0. Importation des bibliothèques

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["image.cmap"] = "gray"

In [ ]:
def find_first_existing(names):
    """Retourne le premier fichier existant dans la liste."""
    for name in names:
        path = Path(name)
        if path.exists():
            return path
    return None


def load_gray(path):
    """Charge une image en niveaux de gris avec OpenCV, puis Pillow en secours."""
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is not None:
        return img

    try:
        from PIL import Image
        return np.array(Image.open(path).convert("L"))
    except Exception as e:
        raise RuntimeError(
            f"Impossible de lire {path}. Si le fichier est en AVIF, "
            "convertissez-le en JPG/PNG ou installez un support AVIF pour Pillow."
        ) from e


def load_color_rgb(path):
    """Charge une image couleur et la retourne en RGB."""
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is not None:
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    try:
        from PIL import Image
        return np.array(Image.open(path).convert("RGB"))
    except Exception as e:
        raise RuntimeError(f"Impossible de lire l'image couleur {path}.") from e

In [ ]:
gray_path = find_first_existing([
    "mountain_gray.jpg",
    "mountain_gray.jpeg",
    "mountain_gray.png",
    "mountain_gray.avif",
])

if gray_path is None:
    raise FileNotFoundError(
        "Image en niveaux de gris introuvable. "
        "Placez mountain_gray.jpg (ou .png/.avif) à côté du notebook."
    )

img = load_gray(gray_path)

print("Image chargée :", gray_path)
print("Type :", img.dtype)
print("Forme :", img.shape)

---
# Exercice 1 — Chargement et affichage d'images

### Objectifs
1. Charger une image en niveaux de gris.
2. Afficher la matrice des pixels et observer la dynamique des valeurs.
3. Identifier les coordonnées d'un pixel particulier, par exemple le pixel de plus forte intensité.
4. Donner la taille de l'image ainsi que les intensités minimale et maximale.

In [ ]:
# 1) Affichage de l'image
plt.figure(figsize=(7, 5))
plt.imshow(img, cmap="gray", vmin=0, vmax=255)
plt.title("Image en niveaux de gris")
plt.axis("off")
plt.show()

In [ ]:
# 2) Matrice des pixels
print("Matrice complète :")
print(img)

print("\nExtrait 10 x 10 :")
print(img[:10, :10])

In [ ]:
# 3) Coordonnées d'un pixel de plus forte intensité
max_value = int(img.max())
y_max, x_max = np.unravel_index(np.argmax(img), img.shape)

print("Coordonnée x :", x_max)
print("Coordonnée y :", y_max)
print("Intensité maximale :", max_value)
print("Valeur vérifiée img[y, x] :", int(img[y_max, x_max]))

In [ ]:
# Questions : taille, minimum et maximum
height, width = img.shape
min_value = int(img.min())
max_value = int(img.max())

print(f"Taille de l'image : {width} x {height} pixels")
print("Intensité minimale :", min_value)
print("Intensité maximale :", max_value)

### Réponse / interprétation
- Une image en niveaux de gris codée sur 8 bits contient en général des intensités comprises entre **0 et 255**.
- **0** représente le noir et **255** le blanc.
- La taille réelle, le minimum et le maximum sont obtenus automatiquement par le code précédent.
- NumPy indexe une image avec l'ordre `img[y, x]` : **y = ligne** et **x = colonne**.

---
# Exercice 2 — Quantification et binarisation

### 1. Réduire le nombre de bits par pixel : 1, 2, 4 et 8 bits

In [ ]:
def quantize_image(image, bits):
    """Quantification uniforme d'une image 8 bits vers 2**bits niveaux."""
    levels = 2 ** bits
    if bits == 8:
        return image.copy()

    q = np.round(image.astype(np.float32) / 255.0 * (levels - 1))
    q = q * (255.0 / (levels - 1))
    return np.clip(q, 0, 255).astype(np.uint8)


bit_depths = [1, 2, 4, 8]
quantized = {b: quantize_image(img, b) for b in bit_depths}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, bits in zip(axes, bit_depths):
    ax.imshow(quantized[bits], cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"{bits} bit(s) — {2**bits} niveaux")
    ax.axis("off")

plt.tight_layout()
plt.show()

### Interprétation de la quantification
- **1 bit** : seulement 2 niveaux, donc perte très forte de détails.
- **2 bits** : 4 niveaux, l'effet de bandes est encore très visible.
- **4 bits** : 16 niveaux, l'image reste reconnaissable avec une perte modérée.
- **8 bits** : 256 niveaux, image originale sans réduction de niveaux.
- Plus le nombre de bits diminue, plus la perte de qualité et les faux contours deviennent visibles.

### 2. Binarisation par seuil fixe et par méthode d'Otsu

In [ ]:
# Seuil fixe
fixed_threshold = 127
_, binary_fixed = cv2.threshold(
    img, fixed_threshold, 255, cv2.THRESH_BINARY
)

# Seuil automatique d'Otsu
otsu_threshold, binary_otsu = cv2.threshold(
    img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

print("Seuil fixe :", fixed_threshold)
print("Seuil calculé par Otsu :", otsu_threshold)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(img, cmap="gray")
axes[0].set_title("Image originale")
axes[0].axis("off")

axes[1].imshow(binary_fixed, cmap="gray", vmin=0, vmax=255)
axes[1].set_title(f"Seuil fixe = {fixed_threshold}")
axes[1].axis("off")

axes[2].imshow(binary_otsu, cmap="gray", vmin=0, vmax=255)
axes[2].set_title(f"Otsu = {otsu_threshold:.0f}")
axes[2].axis("off")

plt.tight_layout()
plt.show()

### Conclusion Exercice 2
Le seuil fixe dépend d'une valeur choisie manuellement. La méthode d'**Otsu** choisit automatiquement un seuil en cherchant une bonne séparation statistique entre deux classes d'intensité. Elle est particulièrement adaptée lorsque l'histogramme présente deux groupes distincts.

---
# Exercice 3 — Calcul et interprétation d'un histogramme

### 1. Calculer et tracer l'histogramme

In [ ]:
hist = cv2.calcHist([img], [0], None, [256], [0, 256])

plt.figure(figsize=(10, 5))
plt.plot(hist)
plt.title("Histogramme de l'image en niveaux de gris")
plt.xlabel("Intensité")
plt.ylabel("Nombre de pixels")
plt.xlim([0, 255])
plt.grid(alpha=0.25)
plt.show()

### 2. Identifier les zones sombres / claires et vérifier une éventuelle sous- ou surexposition

Un histogramme concentré vers **0** indique beaucoup de pixels sombres. Un histogramme concentré vers **255** indique beaucoup de pixels très clairs. Pour parler de saturation réelle, il est préférable d'observer les pixels proches des extrêmes plutôt que d'utiliser arbitrairement 50 et 200.

In [ ]:
# Indicateurs simples proches des extrêmes
dark_limit = 15
bright_limit = 240

very_dark = img < dark_limit
very_bright = img > bright_limit

dark_count = int(np.sum(very_dark))
bright_count = int(np.sum(very_bright))
total = img.size

dark_percent = 100 * dark_count / total
bright_percent = 100 * bright_count / total

print(f"Pixels très sombres (< {dark_limit}) : {dark_count} ({dark_percent:.2f} %)")
print(f"Pixels très clairs (> {bright_limit}) : {bright_count} ({bright_percent:.2f} %)")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(img, cmap="gray")
axes[0].set_title("Image originale")
axes[0].axis("off")

axes[1].imshow(very_dark, cmap="gray")
axes[1].set_title("Masque pixels très sombres")
axes[1].axis("off")

axes[2].imshow(very_bright, cmap="gray")
axes[2].set_title("Masque pixels très clairs")
axes[2].axis("off")

plt.tight_layout()
plt.show()

### 3. Normalisation avec la formule du TP

\[
I'(x,y)=\frac{I(x,y)-\min(I)}{\max(I)-\min(I)}\times255
\]

In [ ]:
img_float = img.astype(np.float32)
i_min = float(img_float.min())
i_max = float(img_float.max())

if i_max == i_min:
    img_normalized = np.zeros_like(img)
else:
    img_normalized = (
        (img_float - i_min) / (i_max - i_min) * 255
    ).astype(np.uint8)

print("Avant normalisation :", int(img.min()), "à", int(img.max()))
print("Après normalisation :", int(img_normalized.min()), "à", int(img_normalized.max()))

In [ ]:
# 4) Visualiser avant et après normalisation
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(img, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Avant normalisation")
axes[0].axis("off")

axes[1].imshow(img_normalized, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Après normalisation")
axes[1].axis("off")

plt.tight_layout()
plt.show()

# Comparaison des histogrammes
plt.figure(figsize=(10, 5))
plt.hist(img.ravel(), bins=256, range=(0, 256), alpha=0.55, label="Avant")
plt.hist(img_normalized.ravel(), bins=256, range=(0, 256), alpha=0.55, label="Après")
plt.title("Histogrammes avant et après normalisation")
plt.xlabel("Intensité")
plt.ylabel("Nombre de pixels")
plt.legend()
plt.show()

### Conclusion Exercice 3
La normalisation étire la dynamique des intensités pour exploiter l'intervalle **[0, 255]**. Si l'image originale utilise seulement une partie de cet intervalle, le contraste global peut être amélioré. Elle ne redistribue toutefois pas les pixels de la même manière qu'une égalisation d'histogramme.

---
# Exercice 4 — Égalisation d'histogramme

### 1. Appliquer une égalisation automatique

In [ ]:
img_equalized = cv2.equalizeHist(img)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(img, cmap="gray")
axes[0].set_title("Image originale")
axes[0].axis("off")

axes[1].imshow(img_equalized, cmap="gray")
axes[1].set_title("Après égalisation")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 2) Comparer les histogrammes avant et après
plt.figure(figsize=(10, 5))
plt.hist(img.ravel(), bins=256, range=(0, 256), alpha=0.55, label="Avant")
plt.hist(img_equalized.ravel(), bins=256, range=(0, 256), alpha=0.55, label="Après")
plt.title("Comparaison des histogrammes")
plt.xlabel("Intensité")
plt.ylabel("Nombre de pixels")
plt.legend()
plt.show()

### 3. Avantages et limites de l'égalisation

**Avantages**
- améliore souvent le contraste global ;
- rend certains détails plus visibles dans les zones peu contrastées ;
- utilise mieux la plage des niveaux de gris.

**Limites**
- peut amplifier le bruit ;
- peut produire un contraste trop fort ou un rendu artificiel ;
- l'égalisation globale ne tient pas compte des différences locales entre régions de l'image.

---
# Exercice 5 — Espaces colorimétriques

Le TP propose une image couleur telle que `flowers.jpg` ou `peppers.png`.

In [ ]:
color_path = find_first_existing([
    "flowers.jpg",
    "flowers.png",
    "peppers.png",
    "peppers.jpg",
])

if color_path is None:
    print(
        "Aucune image couleur trouvée. Ajoutez flowers.jpg ou peppers.png "
        "dans le même dossier, puis relancez cette cellule."
    )
    color_img = None
else:
    color_img = load_color_rgb(color_path)
    print("Image couleur chargée :", color_path, color_img.shape)

In [ ]:
if color_img is not None:
    R = color_img[:, :, 0]
    G = color_img[:, :, 1]
    B = color_img[:, :, 2]

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    axes[0].imshow(color_img)
    axes[0].set_title("Image RGB")
    axes[0].axis("off")

    axes[1].imshow(R, cmap="gray", vmin=0, vmax=255)
    axes[1].set_title("Canal R")
    axes[1].axis("off")

    axes[2].imshow(G, cmap="gray", vmin=0, vmax=255)
    axes[2].set_title("Canal G")
    axes[2].axis("off")

    axes[3].imshow(B, cmap="gray", vmin=0, vmax=255)
    axes[3].set_title("Canal B")
    axes[3].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
if color_img is not None:
    hsv = cv2.cvtColor(color_img, cv2.COLOR_RGB2HSV)

    H = hsv[:, :, 0]
    S = hsv[:, :, 1]
    V = hsv[:, :, 2]

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    axes[0].imshow(H, cmap="hsv")
    axes[0].set_title("H — Teinte")
    axes[0].axis("off")

    axes[1].imshow(S, cmap="gray")
    axes[1].set_title("S — Saturation")
    axes[1].axis("off")

    axes[2].imshow(V, cmap="gray")
    axes[2].set_title("V — Luminosité")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

### Réponse à la question
La composante **H (Hue / Teinte)** est généralement la plus utile pour distinguer différentes zones colorées, car elle représente directement la couleur indépendamment de la luminosité. La composante **S (Saturation)** est aussi utile pour séparer les zones fortement colorées des zones grises ou peu saturées. Lorsque la saturation est très faible, la teinte H devient moins informative.

---
# Exercice 6 — Translation et rotation

### 1. Translation de 50 pixels vers la droite

In [ ]:
h, w = img.shape

translation_matrix = np.float32([
    [1, 0, 50],
    [0, 1, 0]
])

translated = cv2.warpAffine(
    img,
    translation_matrix,
    (w, h),
    flags=cv2.INTER_LINEAR,
    borderValue=0
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img, cmap="gray")
axes[0].set_title("Originale")
axes[0].axis("off")

axes[1].imshow(translated, cmap="gray")
axes[1].set_title("Translation +50 px vers la droite")
axes[1].axis("off")

plt.tight_layout()
plt.show()

### 2. Rotations de 30° et 90° avec interpolation bilinéaire et bicubique

In [ ]:
def rotate_same_size(image, angle, interpolation):
    h, w = image.shape[:2]
    center = (w / 2, h / 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(
        image,
        M,
        (w, h),
        flags=interpolation,
        borderValue=0
    )


rot30_linear = rotate_same_size(img, 30, cv2.INTER_LINEAR)
rot30_cubic = rotate_same_size(img, 30, cv2.INTER_CUBIC)
rot90_linear = rotate_same_size(img, 90, cv2.INTER_LINEAR)
rot90_cubic = rotate_same_size(img, 90, cv2.INTER_CUBIC)

images_rot = [
    (img, "Originale"),
    (rot30_linear, "30° — bilinéaire"),
    (rot30_cubic, "30° — bicubique"),
    (rot90_linear, "90° — bilinéaire"),
    (rot90_cubic, "90° — bicubique"),
]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (im, title) in zip(axes, images_rot):
    ax.imshow(im, cmap="gray")
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

### 3. Comparaison visuelle
- L'interpolation **bilinéaire** calcule chaque nouveau pixel à partir des voisins proches. Elle est rapide et donne généralement un bon compromis.
- L'interpolation **bicubique** utilise davantage de voisins et produit souvent un rendu plus lisse, surtout sur les rotations non multiples de 90°, mais elle est plus coûteuse en calcul.
- Lors d'une rotation avec une taille de sortie identique, certaines parties de l'image peuvent être coupées aux coins.

### Option — Petite mosaïque avant / après / bicubique

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(img, cmap="gray")
axes[0].set_title("Avant")
axes[0].axis("off")

axes[1].imshow(rot30_linear, cmap="gray")
axes[1].set_title("Après — bilinéaire")
axes[1].axis("off")

axes[2].imshow(rot30_cubic, cmap="gray")
axes[2].set_title("Après — bicubique")
axes[2].axis("off")

plt.tight_layout()
plt.show()

---
# Exercice 7 — Filtrage linéaire

### 1. Filtre moyenneur avec k = 3, 5 et 9

In [ ]:
mean_3 = cv2.blur(img, (3, 3))
mean_5 = cv2.blur(img, (5, 5))
mean_9 = cv2.blur(img, (9, 9))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, im, title in zip(
    axes,
    [img, mean_3, mean_5, mean_9],
    ["Originale", "Moyenneur 3x3", "Moyenneur 5x5", "Moyenneur 9x9"]
):
    ax.imshow(im, cmap="gray")
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

### 2. Filtre gaussien avec σ = 1.5 et σ = 3

In [ ]:
gauss_15 = cv2.GaussianBlur(img, (0, 0), sigmaX=1.5)
gauss_3 = cv2.GaussianBlur(img, (0, 0), sigmaX=3)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, im, title in zip(
    axes,
    [img, gauss_15, gauss_3],
    ["Originale", "Gaussien σ = 1.5", "Gaussien σ = 3"]
):
    ax.imshow(im, cmap="gray")
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

### 3. Observation de l'effet sur le bruit et les contours
- Lorsque la taille du noyau moyenneur augmente, le lissage devient plus important.
- Un noyau plus grand réduit davantage le bruit, mais floute aussi davantage les contours et les petits détails.
- Pour le filtre gaussien, une valeur de **σ** plus élevée entraîne un lissage plus fort.
- Le filtre gaussien donne plus de poids aux pixels proches du centre du noyau et produit souvent un lissage plus naturel qu'un simple filtre moyenneur.

---
# Exercice 8 — Filtrage non linéaire

### 1. Ajouter un bruit « poivre et sel »

In [ ]:
def add_salt_pepper_noise(image, amount=0.05, salt_vs_pepper=0.5, seed=42):
    noisy = image.copy()
    rng = np.random.default_rng(seed)

    total_pixels = image.size
    n_noise = int(amount * total_pixels)
    n_salt = int(n_noise * salt_vs_pepper)
    n_pepper = n_noise - n_salt

    # Sel
    ys = rng.integers(0, image.shape[0], n_salt)
    xs = rng.integers(0, image.shape[1], n_salt)
    noisy[ys, xs] = 255

    # Poivre
    yp = rng.integers(0, image.shape[0], n_pepper)
    xp = rng.integers(0, image.shape[1], n_pepper)
    noisy[yp, xp] = 0

    return noisy


noisy_sp = add_salt_pepper_noise(img, amount=0.05)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img, cmap="gray")
axes[0].set_title("Image originale")
axes[0].axis("off")

axes[1].imshow(noisy_sp, cmap="gray")
axes[1].set_title("Bruit poivre et sel")
axes[1].axis("off")

plt.tight_layout()
plt.show()

### 2. Filtre médian avec noyaux 3×3, 5×5 et 9×9

In [ ]:
median_3 = cv2.medianBlur(noisy_sp, 3)
median_5 = cv2.medianBlur(noisy_sp, 5)
median_9 = cv2.medianBlur(noisy_sp, 9)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, im, title in zip(
    axes,
    [noisy_sp, median_3, median_5, median_9],
    ["Bruitée", "Médian 3x3", "Médian 5x5", "Médian 9x9"]
):
    ax.imshow(im, cmap="gray")
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

### 3. Comparaison avec le filtre gaussien

In [ ]:
gaussian_noisy = cv2.GaussianBlur(noisy_sp, (5, 5), sigmaX=1.5)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, im, title in zip(
    axes,
    [img, noisy_sp, median_5, gaussian_noisy],
    ["Originale", "Bruitée", "Médian 5x5", "Gaussien 5x5"]
):
    ax.imshow(im, cmap="gray")
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Comparaison numérique simple avec le PSNR
def psnr(reference, test):
    mse = np.mean(
        (reference.astype(np.float64) - test.astype(np.float64)) ** 2
    )
    if mse == 0:
        return float("inf")
    return 20 * np.log10(255.0 / np.sqrt(mse))


print(f"PSNR image bruitée : {psnr(img, noisy_sp):.2f} dB")
print(f"PSNR médian 3x3    : {psnr(img, median_3):.2f} dB")
print(f"PSNR médian 5x5    : {psnr(img, median_5):.2f} dB")
print(f"PSNR médian 9x9    : {psnr(img, median_9):.2f} dB")
print(f"PSNR gaussien 5x5  : {psnr(img, gaussian_noisy):.2f} dB")

### Conclusion Exercice 8
Le **filtre médian** est particulièrement adapté au bruit impulsionnel « poivre et sel », car il remplace chaque pixel par la médiane de son voisinage et supprime efficacement les valeurs aberrantes tout en conservant relativement bien les contours. Un noyau trop grand peut toutefois supprimer des détails. Le filtre gaussien lisse le bruit, mais il est généralement moins adapté au bruit poivre-et-sel et floute davantage les contours.

---
# Conclusion générale

Ce TP permet de manipuler les principales opérations de base en traitement d'images :
- lecture et analyse des intensités ;
- quantification et binarisation ;
- histogrammes, normalisation et égalisation ;
- espaces de couleur RGB et HSV ;
- translation, rotation et interpolation ;
- filtrage linéaire et non linéaire.

Les résultats numériques exacts dépendent des images utilisées. Exécutez les cellules dans l'ordre pour obtenir les valeurs correspondant à vos fichiers.